<a href="https://colab.research.google.com/github/Edward-Mendoza/EE616---Special-Projects-Spring-2026-CLarkson-University/blob/main/Spring_2026_PCA_Vs_Overall_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# Run this cell first, then restart runtime before continuing
# ============================================================

!pip install --force-reinstall --quiet numpy==1.26.4
!pip install --force-reinstall --quiet scikit-learn==1.3.2
!pip install --force-reinstall --quiet pandas
!pip install --force-reinstall --quiet joblib
!pip install --force-reinstall --quiet opencv-python-headless==4.9.0.80
!pip install --force-reinstall --quiet numpy==1.26.4

In [ ]:
# ============================================================
# CELL 2 — Verify installations
# Restart runtime after Cell 1, then run this cell
# ============================================================

import numpy as np
import pandas as pd
import sklearn
import joblib
import cv2

print("numpy:", np.__version__)        # should be 1.26.4
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__) # should be 1.3.2
print("joblib:", joblib.__version__)
print("cv2:", cv2.__version__)         # should be 4.9.0.80


In [ ]:
# ============================================================
# CELL 3 — Imports
# ============================================================

import numpy as np
import pandas as pd
import io
import os
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab — file upload/download features disabled.")


In [ ]:
# ============================================================
# CELL 4 — Define your 28 metrics and optional custom weights
#
# INSTRUCTIONS:
# The DEFAULT_WEIGHTS dictionary below controls how much
# influence each metric has on the PCA before components
# are extracted. A weight of 1.0 means no change. A weight
# of 2.0 means that metric has double the influence.
#
# To use equal weighting (default), leave all values at 1.0.
# To apply custom weights, change individual values below.
# ============================================================

# --- All 28 OFIQ metrics ---
ALL_METRICS = [
    'BackgroundUniformity.scalar',
    'CompressionArtifacts.scalar',
    'DynamicRange.scalar',
    'EyesVisible.scalar',
    'FaceOcclusionPrevention.scalar',
    'HeadSize.scalar',
    'InterEyeDistance.scalar',
    'LuminanceMean.scalar',
    'LuminanceVariance.scalar',
    'MarginAboveOfTheFaceImage.scalar',
    'MarginBelowOfTheFaceImage.scalar',
    'MouthClosed.scalar',
    'MouthOcclusionPrevention.scalar',
    'NaturalColour.scalar',
    'NoHeadCoverings.scalar',
    'OverExposurePrevention.scalar',
    'SingleFacePresent.scalar',
    'Sharpness.scalar',
    'UnderExposurePrevention.scalar',
    'UnifiedQualityScore.scalar',
]

# --- Component weights for final score calculation ---
# These are applied AFTER PCA runs, weighting each component's
# contribution to the final 0-100 score.
# Leave as None to use variance-proportional weighting automatically.
# To define manually, provide a list with one value per component
# that will be kept — you can adjust after seeing the variance report.
# Example: COMPONENT_WEIGHTS = [0.6, 0.25, 0.15]
COMPONENT_WEIGHTS = None  # set to None for automatic variance-proportional weighting

# --- Metric weights (applied BEFORE PCA) ---
# Controls how much each raw metric influences PCA.
# Default is 1.0 for all — change specific values to emphasize metrics.
METRIC_WEIGHTS = {
    'BackgroundUniformity.scalar'       : 1.0,
    'CompressionArtifacts.scalar'       : 1.0,
    'DynamicRange.scalar'               : 1.0,
    'EyesVisible.scalar'                : 1.0,
    'FaceOcclusionPrevention.scalar'    : 1.0,
    'HeadSize.scalar'                   : 1.0,
    'InterEyeDistance.scalar'           : 1.0,
    'LuminanceMean.scalar'              : 1.0,
    'LuminanceVariance.scalar'          : 1.0,
    'MarginAboveOfTheFaceImage.scalar'  : 1.0,
    'MarginBelowOfTheFaceImage.scalar'  : 1.0,
    'MouthClosed.scalar'                : 1.0,
    'MouthOcclusionPrevention.scalar'   : 1.0,
    'NaturalColour.scalar'              : 1.0,
    'NoHeadCoverings.scalar'            : 1.0,
    'OverExposurePrevention.scalar'     : 1.0,
    'SingleFacePresent.scalar'          : 1.0,
    'Sharpness.scalar'                  : 1.0,
    'UnderExposurePrevention.scalar'    : 1.0,
    'UnifiedQualityScore.scalar'        : 1.0,
}

# --- Variance threshold ---
# PCA will keep however many components are needed to reach this level.
# 0.95 means keep enough components to explain 95% of total variance.
VARIANCE_THRESHOLD = 0.95

In [ ]:
# ============================================================
# CELL 5 — Helper functions
# ============================================================

def upload_file():
    """Uploads a file via Colab dialog or prompts for a local path."""
    if IN_COLAB:
        return files.upload()
    else:
        path = input("Enter path to file: ").strip()
        with open(path, 'rb') as f:
            return {os.path.basename(path): f.read()}


def download_file(filepath):
    """Downloads a file via Colab or prints local path."""
    if IN_COLAB:
        files.download(filepath)
    else:
        print(f"File saved at: {os.path.abspath(filepath)}")


def validate_columns(data, required_cols):
    """Checks that all required columns exist in the dataframe."""
    missing = [c for c in required_cols if c not in data.columns]
    if missing:
        raise ValueError(
            f"The following expected columns are missing from your CSV:\n{missing}"
        )


def apply_metric_weights(data, metrics, metric_weights):
    """
    Multiplies each metric column by its assigned weight.
    Returns a weighted copy — the original dataframe is not modified.
    """
    weighted = data[metrics].copy()
    for col in metrics:
        weight = metric_weights.get(col, 1.0)
        weighted[col] = weighted[col] * weight
        if weight != 1.0:
            print(f"  Applied weight {weight:.2f} to '{col}'")
    return weighted


def compute_component_weights(explained_variance_ratio, user_weights=None):
    """
    Determines the weight for each PCA component when combining into
    the final score.

    If user_weights is None: uses variance-proportional weighting so
    components that captured more variance contribute more to the score.

    If user_weights is provided: normalizes them to sum to 1.0 so the
    final score stays in the 0-100 range regardless of input values.
    """
    n = len(explained_variance_ratio)

    if user_weights is None:
        # Variance-proportional: each component weighted by how much
        # variance it explained relative to the total kept
        weights = explained_variance_ratio / explained_variance_ratio.sum()
        print("Using variance-proportional component weights (automatic).")
    else:
        if len(user_weights) < n:
            # Pad with 1.0 if fewer weights provided than components kept
            user_weights = list(user_weights) + [1.0] * (n - len(user_weights))
        user_weights = np.array(user_weights[:n], dtype=float)
        weights = user_weights / user_weights.sum()
        print("Using manually defined component weights (normalized to sum to 1).")

    for i, (w, v) in enumerate(zip(weights, explained_variance_ratio)):
        print(f"  PC{i+1}: component weight = {w:.4f} | variance explained = {v*100:.2f}%")

    return weights


def print_variance_report(pca, variance_threshold):
    """Prints a detailed explained variance report for all kept components."""
    print("\n" + "="*55)
    print("EXPLAINED VARIANCE REPORT")
    print("="*55)
    cumulative = 0.0
    for i, ratio in enumerate(pca.explained_variance_ratio_):
        cumulative += ratio
        print(
            f"  PC{i+1:>2}: {ratio*100:>6.2f}% variance  |  "
            f"Cumulative: {cumulative*100:>6.2f}%"
        )
    print(f"\n  Total components kept : {pca.n_components_}")
    print(f"  Variance threshold    : {variance_threshold*100:.0f}%")
    print(f"  Total variance kept   : {cumulative*100:.2f}%")
    print("="*55)


def print_correlation_report(data, metrics, score_col='quality_score_0_100'):
    """
    Prints the Pearson correlation between each raw metric and the
    final 0-100 quality score, sorted from strongest to weakest.
    """
    print("\n" + "="*55)
    print("CORRELATION REPORT: Raw Metrics vs Final Score")
    print("="*55)
    correlations = []
    for col in metrics:
        if col in data.columns:
            corr = data[col].corr(data[score_col])
            correlations.append((col, corr))

    # Sort by absolute correlation strength, strongest first
    correlations.sort(key=lambda x: abs(x[1]), reverse=True)

    for col, corr in correlations:
        direction = "↑ positive" if corr >= 0 else "↓ negative (inverse)"
        print(f"  {col:<45} {corr:>+.4f}  {direction}")
    print("="*55)


In [ ]:
# ============================================================
# CELL 6 — PCA Training Module
# ============================================================

def pca_full_training(
    data,
    metrics=ALL_METRICS,
    metric_weights=METRIC_WEIGHTS,
    variance_threshold=VARIANCE_THRESHOLD,
    component_weights=COMPONENT_WEIGHTS,
    output_dir='.'
):
    """
    Trains a single PCA model across all 28 OFIQ metrics and produces
    a 0-100 quality scalar for each image.

    Steps:
    1. Validate all metric columns exist
    2. Apply metric weights (pre-PCA influence control)
    3. Standardize data so all metrics are on equal footing
    4. Run PCA with variance threshold to decide n_components
    5. Compute component weights for final score
    6. Sign-correct components so higher always = better quality
    7. Combine components into a single 0-100 score
    8. Print variance and correlation reports
    9. Save all models and output CSV

    inputs  - data: pandas DataFrame with OFIQ metrics
              metrics: list of metric column names
              metric_weights: dict of per-metric weights (default 1.0)
              variance_threshold: float 0-1, e.g. 0.95 for 95%
              component_weights: list of manual weights or None for automatic
              output_dir: folder to save all output files
    outputs - updated DataFrame, saved models, saved CSV
    """

    os.makedirs(output_dir, exist_ok=True)

    # --- Step 1: Validate columns ---
    print("Validating columns...")
    validate_columns(data, metrics)
    print(f"All {len(metrics)} metric columns found.")

    # --- Step 2: Apply metric weights ---
    print("\nApplying metric weights...")
    weighted_data = apply_metric_weights(data, metrics, metric_weights)

    # --- Step 3: Standardize ---
    # StandardScaler puts all metrics on the same scale before PCA
    # so no metric dominates just because its numbers are larger
    print("\nStandardizing metrics...")
    from sklearn.preprocessing import StandardScaler
    standard_scaler = StandardScaler()
    standardized = standard_scaler.fit_transform(weighted_data)

    # --- Step 4: Run PCA with variance threshold ---
    print(f"\nRunning PCA (keeping components to explain {variance_threshold*100:.0f}% variance)...")
    pca = PCA(n_components=variance_threshold)
    components = pca.fit_transform(standardized)
    n_kept = pca.n_components_
    print(f"PCA complete. {n_kept} components kept.")

    # --- Step 5: Print variance report ---
    print_variance_report(pca, variance_threshold)

    # --- Step 6: Compute component weights ---
    print("\nComputing component weights...")
    comp_weights = compute_component_weights(
        pca.explained_variance_ratio_,
        user_weights=component_weights
    )

    # --- Step 7: Sign correction ---
    # PCA arbitrarily picks component direction. If a component is
    # negatively correlated with the overall quality score, flip it
    # so that higher always = better quality.
    print("\nChecking component sign directions...")
    if 'UnifiedQualityScore.scalar' in data.columns:
        reference = data['UnifiedQualityScore.scalar'].values
        for i in range(n_kept):
            corr = np.corrcoef(components[:, i], reference)[0, 1]
            if corr < 0:
                components[:, i] *= -1
                print(f"  PC{i+1} sign flipped (was negatively correlated with quality reference)")
            else:
                print(f"  PC{i+1} sign OK (positively correlated with quality reference)")
    else:
        print("  UnifiedQualityScore.scalar not found — skipping sign correction.")
        print("  Check correlation report after running to identify any inverted components.")

    # --- Step 8: Store individual PC columns ---
    for i in range(n_kept):
        data[f"pc{i+1}"] = components[:, i]

    # --- Step 9: Combine into raw weighted score ---
    # Multiply each component by its weight and sum
    raw_score = np.zeros(len(data))
    for i in range(n_kept):
        raw_score += components[:, i] * comp_weights[i]

    # --- Step 10: Scale to 0-100 ---
    # MinMaxScaler maps the lowest observed score to 0 and
    # the highest to 100
    min_max_scaler = MinMaxScaler(feature_range=(0, 100))
    data['quality_score_0_100'] = min_max_scaler.fit_transform(
        raw_score.reshape(-1, 1)
    ).flatten()

    print("\nPreview of final quality scores:")
    print(data['quality_score_0_100'].describe().round(2))

    # --- Step 11: Correlation report ---
    print_correlation_report(data, metrics, score_col='quality_score_0_100')

    # --- Step 12: Save everything ---
    print("\nSaving models...")

    # Save PCA model
    pca_path = os.path.join(output_dir, 'full_pca_model.joblib')
    joblib.dump(pca, pca_path)
    download_file(pca_path)

    # Save StandardScaler
    std_scaler_path = os.path.join(output_dir, 'standard_scaler.joblib')
    joblib.dump(standard_scaler, std_scaler_path)
    download_file(std_scaler_path)

    # Save MinMaxScaler
    mm_scaler_path = os.path.join(output_dir, 'minmax_scaler_full.joblib')
    joblib.dump(min_max_scaler, mm_scaler_path)
    download_file(mm_scaler_path)

    # Save component weights
    weights_path = os.path.join(output_dir, 'component_weights.joblib')
    joblib.dump(comp_weights, weights_path)
    download_file(weights_path)

    # Save metric weights used
    metric_weights_path = os.path.join(output_dir, 'metric_weights.joblib')
    joblib.dump(metric_weights, metric_weights_path)
    download_file(metric_weights_path)

    # Save sign correction directions
    sign_corrections_path = os.path.join(output_dir, 'sign_corrections.joblib')
    if 'UnifiedQualityScore.scalar' in data.columns:
        reference = data['UnifiedQualityScore.scalar'].values
        sign_corrections = []
        # Recompute from saved pca to get original signs
        components_check = pca.transform(standardized)
        for i in range(n_kept):
            corr = np.corrcoef(components_check[:, i], reference)[0, 1]
            sign_corrections.append(-1 if corr < 0 else 1)
    else:
        sign_corrections = [1] * n_kept
    joblib.dump(sign_corrections, sign_corrections_path)
    download_file(sign_corrections_path)

    # Save output CSV
    csv_path = os.path.join(output_dir, 'ofiq_full_pca_scores.csv')
    data.to_csv(csv_path, index=False)
    download_file(csv_path)

    print(f"\nAll files saved to: '{os.path.abspath(output_dir)}'")
    print("Training complete.")

    return data


# --- Run training ---
print("Upload your training OFIQ metrics CSV:")
uploaded = upload_file()
fname = list(uploaded.keys())[0]
training_data = pd.read_csv(io.BytesIO(uploaded[fname]))

trained_data = pca_full_training(
    data=training_data,
    metrics=ALL_METRICS,
    metric_weights=METRIC_WEIGHTS,
    variance_threshold=VARIANCE_THRESHOLD,
    component_weights=COMPONENT_WEIGHTS,
    output_dir='pca_output'
)

In [ ]:
# ============================================================
# CELL 7 — Apply to New Images (Validation / Test Module)
# ============================================================

def pca_apply_to_new_images(
    data,
    metrics=ALL_METRICS,
    models_dir='pca_output'
):
    """
    Loads saved models and applies them to new image metrics,
    producing a 0-100 quality score for each image.

    inputs  - data: pandas DataFrame with OFIQ metrics for new images
              metrics: list of metric column names (must match training)
              models_dir: folder containing saved .joblib files
    outputs - DataFrame with quality_score_0_100 column added
    """

    # --- Validate columns ---
    validate_columns(data, metrics)

    # --- Load all saved models ---
    print("Loading saved models...")

    def load_model(filename):
        path = os.path.join(models_dir, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"Model file not found: '{path}'. "
                f"Make sure all files from training are in '{models_dir}'."
            )
        return joblib.load(path)

    pca              = load_model('full_pca_model.joblib')
    standard_scaler  = load_model('standard_scaler.joblib')
    min_max_scaler   = load_model('minmax_scaler_full.joblib')
    comp_weights     = load_model('component_weights.joblib')
    metric_weights   = load_model('metric_weights.joblib')
    sign_corrections = load_model('sign_corrections.joblib')

    print("All models loaded.")

    # --- Apply metric weights ---
    print("\nApplying metric weights...")
    weighted_data = apply_metric_weights(data, metrics, metric_weights)

    # --- Standardize using training scaler ---
    print("Standardizing...")
    standardized = standard_scaler.transform(weighted_data)

    # --- Apply PCA ---
    print("Applying PCA...")
    components = pca.transform(standardized)
    n_kept = pca.n_components_

    # --- Apply sign corrections ---
    for i in range(n_kept):
        components[:, i] *= sign_corrections[i]
        data[f"pc{i+1}"] = components[:, i]

    # --- Combine into weighted score ---
    raw_score = np.zeros(len(data))
    for i in range(n_kept):
        raw_score += components[:, i] * comp_weights[i]

    # --- Scale to 0-100 using training scaler ---
    data['quality_score_0_100'] = min_max_scaler.transform(
        raw_score.reshape(-1, 1)
    ).flatten()

    # --- Clip to 0-100 in case new data falls outside training range ---
    data['quality_score_0_100'] = data['quality_score_0_100'].clip(0, 100)

    print("\nPreview of quality scores for new images:")
    print(data['quality_score_0_100'].describe().round(2))

    # --- Correlation report ---
    print_correlation_report(data, metrics, score_col='quality_score_0_100')

    # --- Save output ---
    output_csv = os.path.join(models_dir, 'new_images_pca_scores.csv')
    data.to_csv(output_csv, index=False)
    download_file(output_csv)

    print("\nDone.")
    return data


# --- Run on new images ---
print("Upload your new images' OFIQ metrics CSV:")
new_uploaded = upload_file()
new_fname = list(new_uploaded.keys())[0]
new_data = pd.read_csv(io.BytesIO(new_uploaded[new_fname]))

# --- Upload saved model files ---
print("\nUpload all 6 saved model files:")
print("  - full_pca_model.joblib")
print("  - standard_scaler.joblib")
print("  - minmax_scaler_full.joblib")
print("  - component_weights.joblib")
print("  - metric_weights.joblib")
print("  - sign_corrections.joblib")
_ = upload_file()

scored_data = pca_apply_to_new_images(
    data=new_data,
    metrics=ALL_METRICS,
    models_dir='.'
)